In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import os
os.chdir('..')
from src.data.sika import CustomImageDataset




# 1. load and transform data
transform = transforms.Compose(
    [transforms.ToTensor()
     ])



trainset = CustomImageDataset(data_path='./data/train', train=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)

testset = CustomImageDataset(data_path='./data/test', train=False, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=2)


In [ ]:
# import f1 score
from sklearn.metrics import f1_score

device = torch.device("cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print(device)

# 2. define CNN-model 
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 7)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 7)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(646176, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        #print(x.shape)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()

# 3. define lossfunction and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

net.to(device)
net.train()

# 4. train model
for epoch in range(10):  # mehrere Epochen durchlaufen

    running_loss = 0.0
    for i, (data) in enumerate(trainloader, 0):
        # Eingaben holen; data ist eine Liste von [inputs, labels]
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        # Parameter-Gradienten auf Null setzen
        optimizer.zero_grad()

        # vorwärts + rückwärts + optimieren
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Statistiken ausgeben
        running_loss += loss.item()

    print('[%d, %5d] loss: %.3f' %
          (epoch + 1, i + 1, running_loss / (i+1)))
    running_loss = 0.0

print('Fertig mit dem Training')

mps


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'src'


In [ ]:

# 5. Modell testen
correct = 0
total = 0
net.eval()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") # wähle die GPU aus, wenn verfügbar.
net.to(device) # verschiebe das Modell auf die GPU.

f1_scores = [] # Liste zum Speichern der F1-Scores

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device) # verschiebe die daten auf die GPU.
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        # calculate f1 score
        f1 = f1_score(labels.cpu(), predicted.cpu(), average='macro') # f1 score Berechnung auf der CPU.
        f1_scores.append(f1) # Füge den F1-Score zur Liste hinzu.

average_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0 # Berechne den Durchschnitt der F1-Scores.

print('Genauigkeit des Netzwerks auf den 200 Testbildern: %d %%' % (
    100 * correct / total))
print('Durchschnittlicher F1-Score: %.3f' % average_f1)

Genauigkeit des Netzwerks auf den 200 Testbildern: 72 %
Durchschnittlicher F1-Score: 0.606
